<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.5-function-calling-and-tools/notebooks/exercises-5_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.5: Function Calling & Tool Use

*Module 5 · Production Applications with GenAI APIs*

> An LLM that can only talk is like a doctor who can diagnose but can't prescribe, operate, or order tests. Function calling lets AI take real actions: query databases, book flights, send emails, and process payments.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-5.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Gemini Function Calling — The Python-Native Way — `part1_gemini_tools.py`
2. Step 2 — OpenAI Tools — JSON Schema + Parallel Calls — `part2_openai_tools.py`
3. Step 3 — Anthropic Tool Use — The Claude Way — `part3_anthropic_tools.py`
4. Step 4 — Advanced Patterns — Chaining, Error Handling, and Schema Design — `part4_advanced_patterns.py`
5. Step 5 — Project: Multi-Provider Tool-Use Chat — `project_tool_chat.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q anthropic>=0.40.0 google-generativeai openai pydantic


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Gemini Function Calling — The Python-Native Way

Pass Python functions directly. Gemini reads the type hints and docstrings.

**`part1_gemini_tools.py`** · _Block 1 of 5_

GEMINI FUNCTION CALLING — Just pass Python functions — Gemini reads


In [ ]:
# =============================================
# GEMINI FUNCTION CALLING
# Just pass Python functions — Gemini reads
# your type hints and docstrings automatically!
# =============================================

import google.generativeai as genai
import os, json

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# ── Define your tools as regular Python functions ──

def get_weather(city: str) -> dict:
    """Get the current weather for a city."""
    # In production, call a real weather API
    data = {
        "hyderabad": {"temp_c": 34, "condition": "Sunny", "humidity": 45},
        "mumbai": {"temp_c": 31, "condition": "Humid", "humidity": 78},
        "delhi": {"temp_c": 38, "condition": "Hot", "humidity": 30},
    }
    return data.get(city.lower(), {"error": f"No data for {city}"})

def search_courses(topic: str, max_price: float = 50000) -> list[dict]:
    """Search for available courses by topic and maximum price in INR."""
    courses = [
        {"name": "GenAI Complete", "price": 29999, "topic": "generative ai"},
        {"name": "Python Mastery", "price": 9999, "topic": "python"},
        {"name": "ML Engineering", "price": 39999, "topic": "machine learning"},
    ]
    return [c for c in courses if topic.lower() in c["topic"] and c["price"] <= max_price]

def calculate_emi(principal: float, rate: float, months: int) -> dict:
    """Calculate EMI (Equated Monthly Installment) for a loan."""
    r = rate / 100 / 12
    emi = principal * r * (1 + r)**months / ((1 + r)**months - 1)
    return {"emi": round(emi, 2), "total": round(emi * months, 2), "interest": round(emi * months - principal, 2)}

# ── Create model with tools ──
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    tools=[get_weather, search_courses, calculate_emi],  # ← just pass functions!
)
chat = model.start_chat()

# ── The AI decides which tool to use ──
messages = [
    "What's the weather like in Hyderabad today?",
    "Show me AI courses under ₹30,000",
    "If I take a ₹30,000 loan at 12% for 6 months, what's the EMI?",
    "Hi, how are you?",  # no tool needed!
]

for msg in messages:
    print(f"\nUser: {msg}")
    response = chat.send_message(msg)
    
    # Check if AI wants to call a function
    for part in response.parts:
        if fn := getattr(part, "function_call", None):
            print(f"  🔧 Tool: {fn.name}({dict(fn.args)})")
            
            # Execute the function
            func = {"get_weather": get_weather, "search_courses": search_courses,
                    "calculate_emi": calculate_emi}[fn.name]
            result = func(**dict(fn.args))
            print(f"  📦 Result: {result}")
            
            # Send result back to AI for natural response
            response = chat.send_message(
                genai.protos.Content(parts=[genai.protos.Part(
                    function_response=genai.protos.FunctionResponse(
                        name=fn.name, response={"result": result}))])
            )
    
    print(f"  AI: {response.text}")


> **What just happened?** We passed 3 Python functions directly to Gemini. It read the type hints ( city: str ), docstrings ("Get current weather"), and parameter names to understand what each tool does. For "What's the weather in Hyderabad?" it automatically called get_weather(city="Hyderabad") . For "Hi, how are you?" it didn't call any tool — just chatted. Gemini's Python-native approach is the simplest: just write regular functions with good docstrings.


### Step 2 · OpenAI Tools — JSON Schema + Parallel Calls

OpenAI uses JSON schemas for tool definitions. Can call MULTIPLE tools in parallel.

**`part2_openai_tools.py`** · _Block 2 of 5_

OPENAI FUNCTION CALLING — JSON schema format + parallel tool calls.


In [ ]:
# =============================================
# OPENAI FUNCTION CALLING
# JSON schema format + parallel tool calls.
# =============================================

from openai import OpenAI
import json

client = OpenAI()

# ── Define tools as JSON schemas ──
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_courses",
            "description": "Search courses by topic and max price in INR",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string"},
                    "max_price": {"type": "number", "default": 50000},
                },
                "required": ["topic"],
            },
        },
    },
]

# ── PARALLEL TOOL CALLS: Ask something that needs 2 tools ──
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "What's the weather in Mumbai AND show me AI courses under 40K"}],
    tools=tools,
)

# The AI may request MULTIPLE tool calls in one response!
tool_calls = response.choices[0].message.tool_calls

if tool_calls:
    print(f"AI wants to call {len(tool_calls)} tools in PARALLEL:\n")
    
    # Execute all tool calls
    tool_results = []
    for tc in tool_calls:
        args = json.loads(tc.function.arguments)
        print(f"  🔧 {tc.function.name}({args})")
        
        # Execute
        if tc.function.name == "get_weather":
            result = get_weather(**args)
        elif tc.function.name == "search_courses":
            result = search_courses(**args)
        else:
            result = {"error": "Unknown function"}
        
        print(f"  📦 {result}")
        tool_results.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(result),
        })
    
    # Send ALL results back at once
    final = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": "What's the weather in Mumbai AND show me AI courses under 40K"},
            response.choices[0].message,  # AI's tool call request
            *tool_results,                # all tool results
        ],
        tools=tools,
    )
    print(f"\n  AI: {final.choices[0].message.content}")

print("""
Key difference from Gemini:
  Gemini: pass Python functions directly (reads type hints)
  OpenAI: define JSON schemas manually (more verbose but explicit)
  
OpenAI's killer feature: PARALLEL tool calls
  "Weather in Mumbai AND courses under 40K" → calls BOTH tools
  simultaneously, not sequentially. Faster for multi-tool queries.
""")


> **What just happened?** OpenAI uses JSON schemas (more verbose than Gemini's Python-native approach but more explicit). The killer feature: parallel tool calls . When the user asked for weather AND courses, OpenAI returned both tool calls in ONE response. We executed both, sent both results back, and the AI combined them into a single coherent answer. This is faster than calling tools one at a time.


### Step 3 · Anthropic Tool Use — The Claude Way

Claude uses a content-block approach with tool_use and tool_result blocks.

**`part3_anthropic_tools.py`** · _Block 3 of 5_

ANTHROPIC TOOL USE — Claude's content-block approach.


In [ ]:
# =============================================
# ANTHROPIC TOOL USE
# Claude's content-block approach.
# =============================================

from anthropic import Anthropic

client = Anthropic()

# ── Define tools (similar schema to OpenAI) ──
tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
            },
            "required": ["city"],
        },
    },
    {
        "name": "calculate_emi",
        "description": "Calculate EMI for a loan",
        "input_schema": {
            "type": "object",
            "properties": {
                "principal": {"type": "number"},
                "rate": {"type": "number", "description": "Annual interest rate %"},
                "months": {"type": "integer"},
            },
            "required": ["principal", "rate", "months"],
        },
    },
]

# ── Chat with tool use ──
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=tools,
    messages=[{"role": "user", "content": "What's the EMI for a ₹50,000 course loan at 10% for 12 months?"}],
)

# Claude's response has content BLOCKS (text + tool_use mixed)
for block in response.content:
    if block.type == "text":
        print(f"  Claude says: {block.text}")
    elif block.type == "tool_use":
        print(f"  🔧 Tool: {block.name}({block.input})")
        
        # Execute
        result = calculate_emi(**block.input)
        print(f"  📦 Result: {result}")
        
        # Send result back
        final = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=[
                {"role": "user", "content": "What's the EMI for a ₹50,000 course loan at 10% for 12 months?"},
                {"role": "assistant", "content": response.content},
                {"role": "user", "content": [
                    {"type": "tool_result", "tool_use_id": block.id,
                     "content": json.dumps(result)}
                ]},
            ],
        )
        print(f"  AI: {final.content[0].text}")


> **What just happened?** Claude uses a content-block approach: the response contains a mix of text blocks and tool_use blocks. When it wants a tool, you get a tool_use block with name + input. You execute it, then send a tool_result block back. The key difference: Claude often explains its reasoning in text BEFORE calling the tool ("I'll calculate the EMI for you..."), making the interaction feel more natural.


### Step 4 · Advanced Patterns — Chaining, Error Handling, and Schema Design

Real production tools are messy. Handle failures, chain multiple calls, and validate everything.

**`part4_advanced_patterns.py`** · _Block 4 of 5_

ADVANCED: Error handling, chaining, validation


In [ ]:
# =============================================
# ADVANCED: Error handling, chaining, validation
# =============================================

from pydantic import BaseModel, Field
from typing import Callable, Any
import traceback, time

class ToolDefinition(BaseModel):
    name: str
    description: str
    function: Callable = Field(exclude=True)
    parameters: dict  # JSON schema
    requires_confirmation: bool = False  # for dangerous actions
    timeout_seconds: float = 10.0

class ToolOrchestrator:
    """Production-grade tool execution with error handling."""
    
    def __init__(self):
        self.tools: dict[str, ToolDefinition] = {}
        self.execution_log = []
    
    def register(self, name: str, func: Callable, description: str,
                 params: dict, dangerous: bool = False):
        """Register a tool with metadata."""
        self.tools[name] = ToolDefinition(
            name=name, description=description, function=func,
            parameters=params, requires_confirmation=dangerous,
        )
    
    def execute(self, tool_name: str, arguments: dict) -> dict:
        """Execute a tool with full error handling and logging."""
        
        if tool_name not in self.tools:
            return {"error": f"Unknown tool: {tool_name}", "success": False}
        
        tool = self.tools[tool_name]
        start = time.time()
        
        # Safety check for dangerous tools
        if tool.requires_confirmation:
            return {
                "needs_confirmation": True,
                "message": f"Action '{tool_name}' requires confirmation. Args: {arguments}",
                "success": False,
            }
        
        # Execute with timeout and error handling
        try:
            result = tool.function(**arguments)
            elapsed = time.time() - start
            
            log_entry = {
                "tool": tool_name, "args": arguments,
                "result": result, "success": True,
                "time_ms": round(elapsed * 1000),
            }
            self.execution_log.append(log_entry)
            return {"result": result, "success": True}
        
        except TypeError as e:
            return {"error": f"Invalid arguments: {e}", "success": False}
        except Exception as e:
            return {"error": f"Tool failed: {e}", "success": False}
    
    def to_gemini_tools(self) -> list:
        """Export tools in Gemini format (Python functions)."""
        return [t.function for t in self.tools.values()]
    
    def to_openai_tools(self) -> list[dict]:
        """Export tools in OpenAI JSON schema format."""
        return [{
            "type": "function",
            "function": {"name": t.name, "description": t.description, "parameters": t.parameters}
        } for t in self.tools.values()]
    
    def to_anthropic_tools(self) -> list[dict]:
        """Export tools in Anthropic format."""
        return [{
            "name": t.name, "description": t.description, "input_schema": t.parameters
        } for t in self.tools.values()]

# ── Register tools ──
orch = ToolOrchestrator()

orch.register("get_weather", get_weather, "Get current weather",
    {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]})

orch.register("search_courses", search_courses, "Search courses by topic",
    {"type": "object", "properties": {"topic": {"type": "string"}, "max_price": {"type": "number"}}, "required": ["topic"]})

orch.register("calculate_emi", calculate_emi, "Calculate EMI",
    {"type": "object", "properties": {"principal": {"type": "number"}, "rate": {"type": "number"}, "months": {"type": "integer"}}, "required": ["principal", "rate", "months"]})

# Test error handling
print("Normal call:")
print(f"  {orch.execute('get_weather', {'city': 'mumbai'})}")

print("\nBad arguments:")
print(f"  {orch.execute('calculate_emi', {'principal': 50000})}")  # missing required args

print("\nUnknown tool:")
print(f"  {orch.execute('hack_server', {})}")

# Export for any provider
print(f"\nGemini format: {len(orch.to_gemini_tools())} tools")
print(f"OpenAI format: {len(orch.to_openai_tools())} tools")
print(f"Anthropic format: {len(orch.to_anthropic_tools())} tools")


> **What just happened?** We built a ToolOrchestrator with: (1) Unified tool registration — register once, export to any provider format (Gemini, OpenAI, Anthropic). (2) Error handling — catches TypeError (bad arguments), unknown tool names, and general exceptions. (3) Safety checks — tools marked dangerous=True require confirmation before execution. (4) Execution logging — tracks every call with args, result, and timing. Define your tools once, use them across all 3 providers.


### Step 5 · Project: Multi-Provider Tool-Use Chat

Complete chat system where the AI uses tools, handles errors, and works with any provider.

**`project_tool_chat.py`** · _Block 5 of 5_

PRODUCTION TOOL-USE CHAT — AI decides which tools to call → your code


In [ ]:
# =============================================
# PRODUCTION TOOL-USE CHAT
# AI decides which tools to call → your code
# executes them → AI uses results to respond.
# Works with Gemini, OpenAI, or Anthropic.
# =============================================

class ToolChat:
    """Chat with automatic tool execution."""
    
    def __init__(self, orchestrator: ToolOrchestrator, provider: str = "gemini"):
        self.orch = orchestrator
        self.provider = provider
        self.max_tool_rounds = 5  # prevent infinite tool loops
    
    def chat(self, message: str, system: str = "") -> str:
        """Send a message, handle all tool calls, return final answer."""
        
        if self.provider == "gemini":
            return self._chat_gemini(message, system)
        elif self.provider == "openai":
            return self._chat_openai(message, system)
        elif self.provider == "anthropic":
            return self._chat_anthropic(message, system)
    
    def _chat_gemini(self, message, system):
        model = genai.GenerativeModel("gemini-2.0-flash",
            tools=self.orch.to_gemini_tools(),
            system_instruction=system or None)
        chat = model.start_chat()
        
        response = chat.send_message(message)
        
        for _ in range(self.max_tool_rounds):
            tool_called = False
            
            for part in response.parts:
                if fn := getattr(part, "function_call", None):
                    tool_called = True
                    print(f"  🔧 {fn.name}({dict(fn.args)})")
                    
                    result = self.orch.execute(fn.name, dict(fn.args))
                    print(f"  📦 {'✅' if result.get('success') else '❌'} {result}")
                    
                    response = chat.send_message(
                        genai.protos.Content(parts=[genai.protos.Part(
                            function_response=genai.protos.FunctionResponse(
                                name=fn.name,
                                response={"result": result.get("result", result.get("error"))}
                            ))]))
            
            if not tool_called:
                break
        
        return response.text
    
    def _chat_openai(self, message, system):
        from openai import OpenAI
        client = OpenAI()
        
        msgs = []
        if system: msgs.append({"role": "system", "content": system})
        msgs.append({"role": "user", "content": message})
        
        for _ in range(self.max_tool_rounds):
            resp = client.chat.completions.create(
                model="gpt-4o", messages=msgs, tools=self.orch.to_openai_tools())
            
            msg = resp.choices[0].message
            
            if not msg.tool_calls:
                return msg.content
            
            msgs.append(msg)
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"  🔧 {tc.function.name}({args})")
                result = self.orch.execute(tc.function.name, args)
                print(f"  📦 {result}")
                msgs.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
        
        return "Max tool rounds reached"
    
    def _chat_anthropic(self, message, system):
        from anthropic import Anthropic
        client = Anthropic()
        msgs = [{"role": "user", "content": message}]
        
        for _ in range(self.max_tool_rounds):
            resp = client.messages.create(
                model="claude-sonnet-4-20250514", max_tokens=1024,
                system=system or "You are helpful.",
                tools=self.orch.to_anthropic_tools(), messages=msgs)
            
            if resp.stop_reason != "tool_use":
                return next((b.text for b in resp.content if b.type == "text"), "")
            
            msgs.append({"role": "assistant", "content": resp.content})
            results = []
            for b in resp.content:
                if b.type == "tool_use":
                    print(f"  🔧 {b.name}({b.input})")
                    result = self.orch.execute(b.name, b.input)
                    print(f"  📦 {result}")
                    results.append({"type": "tool_result", "tool_use_id": b.id,
                                    "content": json.dumps(result)})
            msgs.append({"role": "user", "content": results})
        
        return "Max tool rounds reached"

# ── Use it! ──
tool_chat = ToolChat(orch, provider="gemini")

queries = [
    "What's the weather in Delhi and Hyderabad?",
    "Find AI courses under ₹35,000 and calculate EMI at 10% for 6 months",
    "What's the square root of 144?",  # no tool needed
]

for q in queries:
    print(f"\nUser: {q}")
    answer = tool_chat.chat(q, system="You are a helpful Netsetos assistant.")
    print(f"  AI: {answer[:200]}\n")
    print("─" * 50)


> **What just happened?** We built a complete ToolChat class that: (1) works with all 3 providers (Gemini, OpenAI, Anthropic) through one .chat() method, (2) handles multi-round tool calls (AI calls tool → gets result → calls another tool → gets result → gives final answer), (3) has a safety limit ( max_tool_rounds=5 ) to prevent infinite loops, (4) uses the ToolOrchestrator for unified execution with error handling. The AI can chain tools: find courses → calculate EMI on the price, all in one conversation turn.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
